# `tokenspeed_mla` smoke test

## Example 1 — import, and every public name is the real function (no GPU)

In [ ]:
import importlib.metadata
import tokenspeed_mla

version = importlib.metadata.version("tokenspeed_mla")
# `tokenspeed_mla/__init__.py` hides a failed import: it then binds every public name to a stub that raises when
# called. So "the import worked" is not enough; the names must be the real functions.
assert not hasattr(tokenspeed_mla, "_IMPORT_ERROR"), getattr(tokenspeed_mla, "_IMPORT_ERROR", None)
assert all(getattr(tokenspeed_mla, name).__name__ == name for name in tokenspeed_mla.__all__), tokenspeed_mla.__all__
print(f"tokenspeed_mla {version}:", len(tokenspeed_mla.__all__), "public names, all of them the real functions")

## Example 2 — the helpers that choose a kernel shape, and the GPUs the decode kernels accept (no GPU)

In [ ]:
import cutlass
import torch
from tokenspeed_mla import mla_helpers, utils

assert utils.torch_to_cutlass_dtype(torch.bfloat16) is cutlass.BFloat16
assert utils.cutlass_to_torch_dtype(cutlass.Float16) is torch.float16 and utils.ceil_div(7, 2) == 4
assert [mla_helpers.get_mla_decode_arch(c) for c in ((10, 0), (10, 3))] == ["sm_100", "sm_103"]
# The tuned small FP8 kernel is chosen only on GB200 (10.0) for few heads; everything else gets the 128-wide tiles.
assert mla_helpers.select_mla_decode_tilers(16, 4, is_fp8=True, compute_capability=(10, 0)) == ((64, 128), (64, 256))
assert mla_helpers.select_mla_decode_tilers(128, 4, is_fp8=True, compute_capability=(10, 3)) == ((128, 128), (128, 256))
assert mla_helpers.select_mla_decode_tilers(16, 4, is_fp8=False, compute_capability=(10, 0)) == ((128, 128), (128, 256))
try:
    mla_helpers.get_mla_decode_arch((12, 1))
    raise AssertionError("the decode kernels accepted a GPU they were not written for")
except ValueError as error:
    print("tile shapes chosen as documented; compute capability 12.1 is refused for decode:", error)

## Example 3 — the Triton kernel on the GPU: pack and quantize keys and values to FP8, byte-identical to torch

In [ ]:
import torch
from tokenspeed_mla import get_num_sm, mla_kv_pack_quantize_fp8

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (10, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is not a Blackwell GPU -> this package targets Blackwell GPUs only")
else:
    torch.manual_seed(0)
    tokens, heads = 600, 4
    k_nope = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    k_pe = torch.randn(tokens, 64, device="cuda", dtype=torch.bfloat16)        # one positional part shared by all heads
    v = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    def same_bytes(a, b):
        return bool((a.view(torch.uint8) == b.view(torch.uint8)).all())
    k_fp8, v_fp8 = mla_kv_pack_quantize_fp8(k_nope, k_pe, v)
    assert k_fp8.shape == (tokens, heads, 576) and k_fp8.dtype == v_fp8.dtype == torch.float8_e4m3fn
    expected_k = torch.cat([k_nope, k_pe[:, None, :].expand(tokens, heads, 64)], dim=-1).to(torch.float8_e4m3fn)
    assert same_bytes(k_fp8, expected_k) and same_bytes(v_fp8, v.to(torch.float8_e4m3fn))
    print(f"compute capability {capability[0]}.{capability[1]}, {get_num_sm(torch.device('cuda'))} SMs: FP8 keys and values equal torch's cast, byte for byte")

## Example 4 — the kernel's other documented modes: the e5m2 format, scaling, and launch chaining (PDL)

In [ ]:
import torch
from tokenspeed_mla import mla_kv_pack_quantize_fp8

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (10, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is not a Blackwell GPU -> this package targets Blackwell GPUs only")
else:
    torch.manual_seed(0)
    tokens, heads = 600, 4
    k_nope = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    k_pe = torch.randn(tokens, 64, device="cuda", dtype=torch.bfloat16)        # one positional part shared by all heads
    v = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    def same_bytes(a, b):
        return bool((a.view(torch.uint8) == b.view(torch.uint8)).all())
    joined = torch.cat([k_nope, k_pe[:, None, :].expand(tokens, heads, 64)], dim=-1)
    checked = []
    for label, options, dtype, k_scale, v_scale in (
            ("e5m2", {"fp8_dtype": torch.float8_e5m2}, torch.float8_e5m2, 1.0, 1.0),
            ("scaled", {"k_scale_inv": 0.5, "v_scale_inv": 2.0}, torch.float8_e4m3fn, 0.5, 2.0),
            ("pdl", {"enable_pdl": True}, torch.float8_e4m3fn, 1.0, 1.0)):
        k_fp8, v_fp8 = mla_kv_pack_quantize_fp8(k_nope, k_pe, v, **options)
        assert same_bytes(k_fp8, (joined * k_scale).to(dtype)) and same_bytes(v_fp8, (v * v_scale).to(dtype)), label
        checked.append(label)
    print("byte-identical to torch in every mode:", checked)

## Example 5 — caller-owned output buffers and strided input views

In [ ]:
import torch
from tokenspeed_mla import mla_kv_pack_quantize_fp8

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (10, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is not a Blackwell GPU -> this package targets Blackwell GPUs only")
else:
    torch.manual_seed(0)
    tokens, heads = 600, 4
    k_nope = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    k_pe = torch.randn(tokens, 64, device="cuda", dtype=torch.bfloat16)        # one positional part shared by all heads
    v = torch.randn(tokens, heads, 512, device="cuda", dtype=torch.bfloat16)
    def expected(nope, values):  # the kernel's definition: keys = [nope | k_pe repeated for every head], each cast to e4m3
        keys = torch.cat([nope, k_pe[:, None, :].expand(-1, nope.shape[1], -1)], dim=-1)
        return keys.to(torch.float8_e4m3fn).view(torch.uint8), values.to(torch.float8_e4m3fn).view(torch.uint8)
    # The caller's buffers start as byte 0x7F (NaN in e4m3, which the cast never produces here), so a buffer the kernel
    # did not write cannot pass; torch.empty could hand back GPU memory still holding Examples 3-4's identical bytes.
    k_out = torch.full((tokens, heads, 576), 0x7F, dtype=torch.uint8, device="cuda").view(torch.float8_e4m3fn)
    v_out = torch.full((tokens, heads, 512), 0x7F, dtype=torch.uint8, device="cuda").view(torch.float8_e4m3fn)
    k_fp8, v_fp8 = mla_kv_pack_quantize_fp8(k_nope, k_pe, v, k_out=k_out, v_out=v_out)
    assert k_fp8 is k_out and v_fp8 is v_out, "the function did not return the caller's buffers"
    # Strided inputs from fresh numbers (not Examples 3-4's), so no left-over buffer can already hold the answer.
    wide = torch.randn(tokens, heads, 1024, device="cuda", dtype=torch.bfloat16)
    view, values = wide[:, ::2, :512], wide[:, 1::2, 512:]     # even heads' first halves; odd heads' second halves
    assert not view.is_contiguous() and not values.is_contiguous()
    k_view, v_view = mla_kv_pack_quantize_fp8(view, k_pe, values)
    (want_k, want_v), (want_kv, want_vv) = expected(k_nope, v), expected(view, values)
    assert not ((want_k == 0x7F).any() or (want_v == 0x7F).any()), "the not-written marker is a real output byte"
    compared = {"k_out (the caller's key buffer)": (k_out, want_k), "v_out (the caller's value buffer)": (v_out, want_v),
                "strided keys (columns 0-511)": (k_view[..., :512], want_kv[..., :512]), "strided value output": (v_view, want_vv),
                "positional tail (columns 512-575, from k_pe)": (k_view[..., 512:], want_kv[..., 512:])}
    wrong = [name for name, (got, want) in compared.items() if not torch.equal(got.view(torch.uint8), want)]
    assert not wrong, f"not byte-identical to torch's FP8 cast: {wrong}"
    print("byte-identical to torch's FP8 cast:", "; ".join(compared))